In [1]:
%load_ext autoreload
%autoreload 2

# Summary

Collect logprobs for joke dataset. Would have been nice to do this upfront but used gpt-5-mini. Regardless, it would be nice to have this functionality in general.

In [62]:
import gc
import os
from pathlib import Path
from typing import Optional, Union, Any

In [3]:
repo_parent = Path(".").absolute().parent.parent
os.environ["HF_HOME"] = str(repo_parent/".cache")

In [129]:
# Think we need to import after setting env var if we want custom cache dir.
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from datasets import Dataset, load_dataset
import numpy as np
from huggingface_hub import login, HfApi

from aeon.secrets import SecretManager

In [5]:
name = "Qwen/Qwen3-8B-Base"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [6]:
_ = model.to("cuda")

In [7]:
assert model.device.type == 'cuda'

In [144]:
def get_logprobs(text: str, model, tokenizer, k: int = 10, i2w: Optional[dict] = None):
    """For an existing text sequence (can be LLM-generated, human-written, whatever),
    get some model's logprobs for each token. Essentially shows us how surprising each
    token was.
    
    Parameters
    ----------
    k : int
        Number of most probably tokens to return logprobs for at each step.
    i2w : dict or NoneType
        Maps tokenizer token index (int) to token (str). If not provided, we will
        construct it from the `tokenizer` arg. (Just saves a little time to not have
        to iterate over the whole vocab an extra time on every batch since we want to
        run this func on any inputs.)
    """
    vocab = tokenizer.get_vocab()
    i2w = i2w or {i: word for word, i in tokenizer.get_vocab().items()}
    tokens = tokenizer.tokenize(text)
    # We will use these as labels later.
    token_idx = torch.tensor([vocab[t] for t in tokens], device=model.device)
    # list[str]
    sequences = [
        tokenizer.convert_tokens_to_string(tokens[:i])
        for i in np.arange(len(tokens))
    ]

    all_inputs = []
    for seq in sequences:
        messages = [{"role": "user", "content": seq}]
        inputs = tokenizer.apply_chat_template(
        	messages,
            continue_final_message=True,
        	add_generation_prompt=False,
        	tokenize=True,
        	return_dict=True,
        	return_tensors="pt",
        )
        inputs["input_ids"] = inputs["input_ids"].squeeze()
        inputs["attention_mask"] = inputs["attention_mask"].squeeze()
        all_inputs.append(inputs)
    padded_inputs = tokenizer.pad(all_inputs, padding=True, padding_side="left", 
                                  max_length=len(tokens)).to(model.device)
    
    outputs = model.generate(**padded_inputs, max_new_tokens=1,
                             return_dict_in_generate=True, output_scores=True)

    # outputs.scores has len max_new_tokens which is always 1 in our case.
    # Just pull out the relevant bit for easy handling.
    # shape: (bs, vocab_size)
    scores = outputs.scores[0]
    logprobs_allrows = scores.log_softmax(dim=-1)
    # logprob for correct next token for each row.
    label_logprobs = logprobs_allrows[torch.arange(logprobs_allrows.shape[0]).to(model.device), token_idx]
    
    # Get index of top 10 logprobs for each row
    idx_allrows = logprobs_allrows.argsort(dim=-1, descending=True)
    label_rank = (idx_allrows == token_idx.unsqueeze(-1)).nonzero()[:, -1]
    idx_topk = idx_allrows[:, :k]
    logprobs_topk = logprobs_allrows.gather(-1, idx_topk)

    res = []
    for label, label_logprob, rank, idx, logprobs in zip(
        tokens, label_logprobs, label_rank, idx_topk, logprobs_topk
    ):
        probs = logprobs.exp()
        item = {
            "label": label,
            "label_prob": label_logprob.exp().item(),
            "label_rank": rank.item(),
            "label_logprob": label_logprob.item(),
            "top_k_probs": {
                i2w[i.item()]: prob.item() for i, prob in zip(idx, logprobs)
            },
            "top_k_logprobs": {
                i2w[i.item()]: logprob.item() for i, logprob in zip(idx, logprobs)
            },
        }
        res.append(item)
    return res

In [127]:
vocab = tokenizer.get_vocab()
i2w = {i: word for word, i in vocab.items()}

In [124]:
res = get_logprobs("The sky is blue and grass is green.", model, tokenizer, i2w=i2w)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


In [125]:
res

[{'label': 'The',
  'label_prob': 0.009832409210503101,
  'label_rank': 13,
  'label_logprob': -4.622071266174316,
  'top_k_probs': {'#': 0.028008118271827698,
   'Re': 0.027387190610170364,
   '1': 0.026034632697701454,
   '===': 0.022143740206956863,
   '2': 0.019353127107024193,
   '====Ċ': 0.015335707925260067,
   '==': 0.014899643138051033,
   'ĠĠĠ': 0.01483980007469654,
   'Ġ*': 0.014062963426113129,
   '0': 0.012476739473640919},
  'top_k_logprobs': {'#': -3.575260877609253,
   'Re': -3.597679853439331,
   '1': -3.648327589035034,
   '===': -3.8102004528045654,
   '2': -3.94490122795105,
   '====Ċ': -4.1775712966918945,
   '==': -4.206418037414551,
   'ĠĠĠ': -4.210442543029785,
   'Ġ*': -4.2642107009887695,
   '0': -4.383889198303223}},
 {'label': 'Ġsky',
  'label_prob': 0.00010148722503799945,
  'label_rank': 1452,
  'label_logprob': -9.195577621459961,
  'top_k_probs': {'Ġuser': 0.026565412059426308,
   'Ġfollowing': 0.018607156351208687,
   'Ġ': 0.014498427510261536,
   'Ġnum

In [131]:
ds = load_dataset("hmamin/extract_jokes")

README.md:   0%|          | 0.00/531 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.94M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22929 [00:00<?, ? examples/s]

In [140]:
df = ds['train'].to_pandas()

In [142]:
df.tail()

,prompt,joke,subtext,unfunny_variant,web_scraper_order,transcript_link,transcript_link_href
22924,Any better short-term investment ideas?,"Why don’t you name a better way to make $6,000...",Foolish financial decisions can be rationalize...,I rationalized a bad financial loss as a way t...,1686242983-419,John Mulaney: Baby J (2023) | Transcript,https://scrapsfromtheloft.com/comedy/john-mula...
22925,How has recovery changed your self-image?,It’s weird to be a recovering drug addict... S...,Surviving personal harm can shift priorities a...,"Having survived addiction, I'm less concerned ...",1686242983-419,John Mulaney: Baby J (2023) | Transcript,https://scrapsfromtheloft.com/comedy/john-mula...
22926,Any awkward public parenting moments?,I was in a museum in Detroit with my son and n...,Parents can be confronted with reminders of pa...,"While changing my baby in a museum restroom, I...",1686242983-419,John Mulaney: Baby J (2023) | Transcript,https://scrapsfromtheloft.com/comedy/john-mula...
22927,Do you remember things you said on drugs?,"I gave an interview to GQ December 15th, 2020 ...",Substance use impairs memory and leads to inco...,I don't remember certain interviews I apparent...,1686242983-419,John Mulaney: Baby J (2023) | Transcript,https://scrapsfromtheloft.com/comedy/john-mula...
22928,"If you had a talk show, what would it be like?","GQ asked if I'd want my own talk show. I said,...",Some talk show concepts are oddly specific or ...,"I once thought about two talk show concepts, i...",1686242983-419,John Mulaney: Baby J (2023) | Transcript,https://scrapsfromtheloft.com/comedy/john-mula...


# Idea:

maybe should allow excluding the first n tokens from the logprob exercise. Like I could pass in f"{prompt}\n{joke}" and keep {prompt} fixed, so the first token of joke is conditioned on prompt rather than on nothing.